In [0]:
# =============================================================
# 04_silver_upsert — Delta Lake MERGE (upsert) into Silver
# Author: oakville3456
# Branch: feature/priority3-upsert
# Purpose: Insert new rows, update changed rows — no duplicates
# =============================================================

BRONZE = "abfss://bronze@saretailsalesdev.dfs.core.windows.net/sales"
SILVER = "abfss://silver@saretailsalesdev.dfs.core.windows.net/sales"

# Read the full Bronze table
bronze_df = spark.read.format("delta").load(BRONZE)

print(f"✅ Bronze rows loaded: {bronze_df.count()}")
bronze_df.printSchema()


# Read the full SILVER table
SILVER_df = spark.read.format("delta").load(SILVER)

print(f"✅ SILVER rows loaded: {SILVER_df.count()}")
SILVER_df.printSchema()

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Clean Bronze
silver_df = (
    bronze_df
    .filter(F.col("order_id").isNotNull())
    .filter(F.col("price").isNotNull())
    .filter(F.col("price") > 0)
    .withColumn("order_date", F.try_to_date("order_date"))
    .withColumn("revenue",    F.col("quantity") * F.col("price"))
    .withColumn("_updated_at", F.current_timestamp())
)

# Deduplicate — keep the latest row per order_id
window = Window.partitionBy("order_id").orderBy(F.col("_ingested_at").desc())

silver_df = (
    silver_df
    .withColumn("_row_num", F.row_number().over(window))
    .filter(F.col("_row_num") == 1)
    .drop("_row_num")
)

# Verify no duplicates remain
dup_count = silver_df.groupBy("order_id").count().filter("count > 1").count()
print(f"✅ Duplicate order_ids remaining: {dup_count}  ← must be 0")
print(f"✅ Clean rows ready to upsert: {silver_df.count()}")

In [0]:
# MERGE (upsert) into Silver
if DeltaTable.isDeltaTable(spark, SILVER):
    silver_table = DeltaTable.forPath(spark, SILVER)

    (
        silver_table.alias("target")
        .merge(
            silver_df.alias("source"),
            "target.order_id = source.order_id"
        )
        .whenMatchedUpdate(set={
            "store_id":     "source.store_id",
            "product":      "source.product",
            "quantity":     "source.quantity",
            "price":        "source.price",
            "order_date":   "source.order_date",
            "customer_id":  "source.customer_id",
            "revenue":      "source.revenue",
            "_ingested_at": "source._ingested_at",
            "_source_file": "source._source_file",
        })
        .whenNotMatchedInsertAll()
        .execute()
    )
    print("✅ MERGE complete")

else:
    silver_df.write.format("delta").mode("overwrite").save(SILVER)
    print("✅ Silver table created (first run)")

final_count = spark.read.format("delta").load(SILVER).count()
print(f"✅ Silver rows after upsert: {final_count}")

In [0]:
# Cell 4 — Audit: how many rows were UPDATED vs INSERTED?
from delta.tables import DeltaTable

# Read Delta transaction history
silver_table = DeltaTable.forPath(spark, SILVER)
history = silver_table.history(1)  # last operation only

history.select(
    "version",
    "timestamp",
    "operation",
    "operationMetrics"
).show(truncate=False)

In [0]:
# Cell 5 — re-run to confirm clean Silver
silver = spark.read.format("delta").load(SILVER)

total        = silver.count()
null_orderid = silver.filter(F.col("order_id").isNull()).count()
null_price   = silver.filter(F.col("price").isNull()).count()
neg_price    = silver.filter(F.col("price") <= 0).count()
null_date    = silver.filter(F.col("order_date").isNull()).count()
duplicates   = silver.groupBy("order_id").count().filter("count > 1").count()

print(f"✅ Total Silver rows      : {total}")
print(f"✅ Duplicate order_ids    : {duplicates}  ← must be 0")
print(f"⚠️  Null order_ids        : {null_orderid}")
print(f"⚠️  Null prices           : {null_price}")
print(f"⚠️  Negative prices       : {neg_price}")
print(f"⚠️  Null dates (bad input): {null_date}")

In [0]:
# Cell 6 — Investigate duplicates in Silver
silver = spark.read.format("delta").load(SILVER)

# Show duplicate order_ids and how many times each appears
dups = (
    silver
    .groupBy("order_id")
    .count()
    .filter("count > 1")
    .orderBy("count", ascending=False)
)

print(f"Duplicate order_id groups: {dups.count()}")
dups.show(10)

# Show actual duplicate rows for one order_id
first_dup_id = dups.first()["order_id"]
print(f"\nSample — all rows for order_id {first_dup_id}:")
silver.filter(F.col("order_id") == first_dup_id).show(truncate=False)

In [0]:
# Cell 7 — Rebuild Silver by deduplicating existing data
from delta.tables import DeltaTable
from pyspark.sql.window import Window

silver = spark.read.format("delta").load(SILVER)

# Keep only the latest row per order_id
window = Window.partitionBy("order_id").orderBy(F.col("_ingested_at").desc())

silver_clean = (
    silver
    .withColumn("_row_num", F.row_number().over(window))
    .filter(F.col("_row_num") == 1)
    .drop("_row_num")
)

# Overwrite Silver with clean data
(
    silver_clean
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(SILVER)
)

# Verify
final = spark.read.format("delta").load(SILVER)
dups  = final.groupBy("order_id").count().filter("count > 1").count()
print(f"✅ Silver rows after dedup : {final.count()}")
print(f"✅ Duplicate order_ids     : {dups}  ← must be 0")

In [0]:
# Cell 8 — Idempotency test: run MERGE again, Silver must not change

before_count = spark.read.format("delta").load(SILVER).count()

# Re-run the exact same MERGE from Cell 3
silver_table = DeltaTable.forPath(spark, SILVER)

(
    silver_table.alias("target")
    .merge(
        silver_df.alias("source"),
        "target.order_id = source.order_id"
    )
    .whenMatchedUpdate(set={
        "store_id":     "source.store_id",
        "product":      "source.product",
        "quantity":     "source.quantity",
        "price":        "source.price",
        "order_date":   "source.order_date",
        "customer_id":  "source.customer_id",
        "revenue":      "source.revenue",
        "_ingested_at": "source._ingested_at",
        "_source_file": "source._source_file",
    })
    .whenNotMatchedInsertAll()
    .execute()
)

after_count = spark.read.format("delta").load(SILVER).count()

print(f"✅ Silver rows before : {before_count}")
print(f"✅ Silver rows after  : {after_count}")
print(f"✅ Difference         : {after_count - before_count}  ← must be 0")

if after_count == before_count:
    print("✅ MERGE is idempotent — safe to rerun anytime!")
else:
    print("❌ Something wrong — row count changed!")